In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np


In [2]:
pwd

'/home/saishyam/Protein_dynamics/Dynamic_properties/Protein_Cell_tissue_confidence'

In [3]:
pip install bioservices tqdm


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
ls

Dataset_filtering_prot_seq_add.ipynb  tissues_with_uniprot.tsv
gene_attribute_matrix.txt


In [5]:
conf_data = pd.read_csv("gene_attribute_matrix.txt", sep="\t")

/tmp/ipykernel_3644627/3456435742.py:1: DtypeWarning: Columns (2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149,150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165,166,167,168,169,170,171,172,173,174,175,176,177,178,179,180,181,182,183,184,185,186,187,188,189,190,191,192,193,194,195,196,197,198,199,200,201,202,203,204,205,206,207,208,209,210,211,212,213,214,215,216,217,218,219,220,221,222,223,224,225,226,227,228,229,230,231,232,233,234,235,236,237,238,239,240,241,242,243,244,245) have mixed types. Specify dtype option on import or set low_memory

In [6]:
conf_data

,#,#.1,Tissue/Cell Type,fetus,placenta,embryonic structure,organism form,internal female genital organ,viscus,alimentary canal,...,culture condition:cd8+ cell,culture condition:cd4+ cell,bone marrow cell,megakaryoblast,blood platelet,megakaryocyte,blood plasma,trachea,larynx,ear
0,#,#,BTO,BTO:0000449,BTO:0001078,BTO:0000174,BTO:0000284,BTO:0003099,BTO:0001491,BTO:0000058,...,BTO:0004410,BTO:0005453,BTO:0004850,BTO:0001164,BTO:0000132,BTO:0000843,BTO:0000131,BTO:0001388,BTO:0001208,BTO:0000368
1,GeneSym,Ensemble Acc,GeneID/NA,na,na,na,na,na,na,na,...,na,na,na,na,na,na,na,na,na,na
2,ZNRD1,ENSP00000383503,30834,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,ZPR1,ENSP00000227322,8882,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,AACS,ENSP00000324842,65985,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15502,FGL2,ENSP00000248598,10875,0.0,0.0,0.0,0.0,0.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
15503,CSTA,ENSP00000264474,1475,0.0,0.0,0.0,0.0,0.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
15504,S100A8,ENSP00000357721,6279,0.0,0.0,0.0,0.0,1.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
15505,HLA-DMB,ENSP00000378723+ENSP00000393646,3109,0.0,0.0,0.0,0.0,0.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [8]:
import requests
import pandas as pd
from tqdm import tqdm
import time

# ============================================================
# LOAD FILE
# ============================================================

# conf_data = pd.read_csv("your_file.tsv", sep="\t", header=None)

# ============================================================
# FIX HEADER STRUCTURE
# ============================================================

new_cols = conf_data.iloc[1].tolist()
new_cols[0] = "GeneSym"
new_cols[1] = "ENSP"
new_cols[2] = "GeneID"

conf_data = conf_data.iloc[2:].copy()
conf_data.columns = new_cols

# ============================================================
# EXTRACT ENSP IDS
# ============================================================

def clean_ensp(x):
    if pd.isna(x):
        return None
    return str(x).split("+")[0]

conf_data["ENSP_clean"] = conf_data["ENSP"].apply(clean_ensp)

# ============================================================
# BATCH MAP ENSP -> UNIPROT
# ============================================================

def batch_ensp_to_uniprot(ensp_list, batch_size=500, poll_interval=1.0, max_polls=30):
    BASE = "https://rest.uniprot.org"
    results = {}
    batches = [ensp_list[i : i + batch_size] for i in range(0, len(ensp_list), batch_size)]

    for batch in tqdm(batches, desc="Mapping batches"):
        # 1. Submit job
        r = requests.post(
            f"{BASE}/idmapping/run",
            data={
                "from": "Ensembl_Protein",
                "to":   "UniProtKB",
                "ids":  ",".join(batch),
            },
            timeout=30,
        )
        r.raise_for_status()
        job_id = r.json()["jobId"]

        # 2. Poll until complete
        for _ in range(max_polls):
            status = requests.get(f"{BASE}/idmapping/status/{job_id}", timeout=15)
            status.raise_for_status()
            info = status.json()
            if "results" in info or "failedIds" in info:
                break
            time.sleep(poll_interval)
        else:
            print(f"Warning: job {job_id} timed out — skipping batch")
            continue

        # 3. Fetch TSV results
        res = requests.get(
            f"{BASE}/idmapping/results/{job_id}",
            params={"format": "tsv", "fields": "accession"},
            timeout=30,
        )
        res.raise_for_status()

        lines = res.text.strip().split("\n")[1:]  # skip header
        for line in lines:
            parts = line.split("\t")
            if len(parts) >= 2:
                ensp, acc = parts[0].strip(), parts[1].strip()
                results.setdefault(ensp, acc)

    # Unmapped IDs get None
    for ensp in ensp_list:
        results.setdefault(ensp, None)

    return results

# ============================================================
# RUN MAPPING
# ============================================================

unique_ensp = conf_data["ENSP_clean"].dropna().unique().tolist()
ensp_to_uniprot = batch_ensp_to_uniprot(unique_ensp)
conf_data["UniProt_ID"] = conf_data["ENSP_clean"].map(ensp_to_uniprot)

# ============================================================
# SAVE
# ============================================================

conf_data.to_csv("tissues_with_uniprot.tsv", sep="\t", index=False)
print(conf_data[["GeneSym", "ENSP", "UniProt_ID"]].head())

Mapping batches: 100%|██████████| 32/32 [02:25<00:00,  4.54s/it]


   GeneSym             ENSP UniProt_ID
2    ZNRD1  ENSP00000383503     Q9P1U0
3     ZPR1  ENSP00000227322     O75312
4     AACS  ENSP00000324842     Q86V21
5  ABHD17B  ENSP00000366240     Q5VST6
6    ABHD8  ENSP00000247706     Q96I13


In [12]:
conf_data.loc[:,  "UniProt_ID"].value_counts().sum()

np.int64(780)

In [ ]:
conf_data.loc[:,"UniProt_ID"].isna().sum()

np.int64(15478)

['ENSP00000383503',
 'ENSP00000227322',
 'ENSP00000324842',
 'ENSP00000366240',
 'ENSP00000247706',
 'ENSP00000331465',
 'ENSP00000354414',
 'ENSP00000269829',
 'ENSP00000324598',
 'ENSP00000347648',
 'ENSP00000310015',
 'ENSP00000219478',
 'ENSP00000359740',
 'ENSP00000380019',
 'ENSP00000252979',
 'ENSP00000379231',
 'ENSP00000338572',
 'ENSP00000379847',
 'ENSP00000305804',
 'ENSP00000228289',
 'ENSP00000231749',
 'ENSP00000322915',
 'ENSP00000312222',
 'ENSP00000326921',
 'ENSP00000216923',
 'ENSP00000291900',
 'ENSP00000337475',
 'ENSP00000266529',
 'ENSP00000351052',
 'ENSP00000404580',
 'ENSP00000359364',
 'ENSP00000204279',
 'ENSP00000327821',
 'ENSP00000351137',
 'ENSP00000327716',
 'ENSP00000324203',
 'ENSP00000267973',
 'ENSP00000006526',
 'ENSP00000351446',
 'ENSP00000383599',
 'ENSP00000296679',
 'ENSP00000263150',
 'ENSP00000335434',
 'ENSP00000263461',
 'ENSP00000326379',
 'ENSP00000238497',
 'ENSP00000309457',
 'ENSP00000318629',
 'ENSP00000381282',
 'ENSP00000261776',


In [8]:
conf_data

,#,#.1,Tissue/Cell Type,fetus,placenta,embryonic structure,organism form,internal female genital organ,viscus,alimentary canal,...,culture condition:cd8+ cell,culture condition:cd4+ cell,bone marrow cell,megakaryoblast,blood platelet,megakaryocyte,blood plasma,trachea,larynx,ear
0,#,#,BTO,BTO:0000449,BTO:0001078,BTO:0000174,BTO:0000284,BTO:0003099,BTO:0001491,BTO:0000058,...,BTO:0004410,BTO:0005453,BTO:0004850,BTO:0001164,BTO:0000132,BTO:0000843,BTO:0000131,BTO:0001388,BTO:0001208,BTO:0000368
1,GeneSym,Ensemble Acc,GeneID/NA,na,na,na,na,na,na,na,...,na,na,na,na,na,na,na,na,na,na
2,ZNRD1,ENSP00000383503,30834,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,ZPR1,ENSP00000227322,8882,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,AACS,ENSP00000324842,65985,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15502,FGL2,ENSP00000248598,10875,0.0,0.0,0.0,0.0,0.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
15503,CSTA,ENSP00000264474,1475,0.0,0.0,0.0,0.0,0.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
15504,S100A8,ENSP00000357721,6279,0.0,0.0,0.0,0.0,1.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
15505,HLA-DMB,ENSP00000378723+ENSP00000393646,3109,0.0,0.0,0.0,0.0,0.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [9]:
conf_data

,#,#.1,Tissue/Cell Type,fetus,placenta,embryonic structure,organism form,internal female genital organ,viscus,alimentary canal,...,culture condition:cd8+ cell,culture condition:cd4+ cell,bone marrow cell,megakaryoblast,blood platelet,megakaryocyte,blood plasma,trachea,larynx,ear
0,#,#,BTO,BTO:0000449,BTO:0001078,BTO:0000174,BTO:0000284,BTO:0003099,BTO:0001491,BTO:0000058,...,BTO:0004410,BTO:0005453,BTO:0004850,BTO:0001164,BTO:0000132,BTO:0000843,BTO:0000131,BTO:0001388,BTO:0001208,BTO:0000368
1,GeneSym,Ensemble Acc,GeneID/NA,na,na,na,na,na,na,na,...,na,na,na,na,na,na,na,na,na,na
2,ZNRD1,ENSP00000383503,30834,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,ZPR1,ENSP00000227322,8882,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,AACS,ENSP00000324842,65985,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15502,FGL2,ENSP00000248598,10875,0.0,0.0,0.0,0.0,0.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
15503,CSTA,ENSP00000264474,1475,0.0,0.0,0.0,0.0,0.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
15504,S100A8,ENSP00000357721,6279,0.0,0.0,0.0,0.0,1.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
15505,HLA-DMB,ENSP00000378723+ENSP00000393646,3109,0.0,0.0,0.0,0.0,0.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [18]:

# ============================================================
# ENSP -> UniProt (BATCH VERSION - RELIABLE)
# ============================================================

import pandas as pd
import requests
import time
from io import StringIO
from tqdm.auto import tqdm

# ============================================================
# CLEAN ENSP IDs
# ============================================================

conf_data["ENSP"] = (
    conf_data["ENSP"]
    .astype(str)
    .str.split("+")
    .str[0]
    .str.strip()
)

# ============================================================
# REMOVE OLD UniProt COLUMNS
# ============================================================

conf_data = conf_data.drop(
    columns=[c for c in conf_data.columns if "UniProt" in str(c)],
    errors="ignore"
)

# ============================================================
# UNIQUE ENSP IDS
# ============================================================

ensp_ids = (
    conf_data["ENSP"]
    .dropna()
    .unique()
    .tolist()
)

print(f"Unique ENSP IDs: {len(ensp_ids)}")

# ============================================================
# BATCH SETTINGS
# ============================================================

BATCH_SIZE = 500

batches = [
    ensp_ids[i:i+BATCH_SIZE]
    for i in range(0, len(ensp_ids), BATCH_SIZE)
]

print(f"Total batches: {len(batches)}")

# ============================================================
# FUNCTION
# ============================================================

def map_batch(batch_ids):
    BASE = "https://rest.uniprot.org"

    # Submit job
    response = requests.post(
        f"{BASE}/idmapping/run",
        data={
            "from": "Ensembl_Protein",
            "to":   "UniProtKB",
            "ids":  ",".join(batch_ids),
        },
        timeout=30,
    )
    response.raise_for_status()
    job_id = response.json()["jobId"]

    # Poll until complete (max 120 s)
    for _ in range(60):
        status_resp = requests.get(
            f"{BASE}/idmapping/status/{job_id}",
            timeout=15,
        )
        status_resp.raise_for_status()
        info = status_resp.json()
        if (
            info.get("jobStatus") == "FINISHED"
            or "results" in info
            or "failedIds" in info
        ):
            break
        time.sleep(2)
    else:
        raise TimeoutError(f"Job {job_id} did not finish in time")

    # Download TSV results
    r = requests.get(
        f"{BASE}/idmapping/uniprotkb/results/stream/{job_id}",
        params={"format": "tsv", "fields": "accession"},
        timeout=60,
    )
    r.raise_for_status()

    if not r.text.strip():
        return pd.DataFrame(columns=["ENSP", "UniProt_ID"])

    df = pd.read_csv(StringIO(r.text), sep="\t")

    if len(df) == 0:
        return pd.DataFrame(columns=["ENSP", "UniProt_ID"])

    df = df.rename(columns={"From": "ENSP", "Entry": "UniProt_ID"})

    keep = [c for c in ["ENSP", "UniProt_ID"] if c in df.columns]
    return df[keep]

# ============================================================
# RUN ALL BATCHES
# ============================================================

all_mapping = []

for batch in tqdm(batches, desc="Processing batches"):
    try:
        batch_df = map_batch(batch)
        all_mapping.append(batch_df)
    except Exception as e:
        print("Batch failed:", e)

# ============================================================
# CONCAT & DEDUPLICATE
# ============================================================

if all_mapping:
    mapping_df = pd.concat(all_mapping, ignore_index=True)
    mapping_df = mapping_df.drop_duplicates(subset=["ENSP"])
    print(mapping_df.head())
    print(f"\nTotal mappings: {len(mapping_df)}")
else:
    mapping_df = pd.DataFrame(columns=["ENSP", "UniProt_ID"])
    print("No mappings found")

# ============================================================
# MERGE BACK
# ============================================================

conf_data = conf_data.merge(
    mapping_df,
    on="ENSP",
    how="left"
)

# ============================================================
# SUMMARY
# ============================================================

mapped = conf_data["UniProt_ID"].notna().sum()
print(f"\nMapped proteins: {mapped} / {len(conf_data)}")

unmapped = conf_data[conf_data["UniProt_ID"].isna()]
print(f"Unmapped proteins: {len(unmapped)}")

display(unmapped[["GeneSym", "ENSP"]].head())

# ============================================================
# FINAL OUTPUT
# ============================================================

display(conf_data.head())

# ============================================================
# SAVE
# ============================================================

conf_data.to_csv("TISSUES_with_UniProt.csv", index=False)
print("Saved to TISSUES_with_UniProt.csv")


Unique ENSP IDs: 15499
Total batches: 31


Processing batches: 100%|██████████| 31/31 [02:32<00:00,  4.92s/it]

              ENSP UniProt_ID
0  ENSP00000354414     Q7L945
1  ENSP00000269829     Q6NX49
2  ENSP00000324598     Q8NDQ6
3  ENSP00000347648     Q96ME7
4  ENSP00000310015     Q9BUB4

Total mappings: 13705

Mapped proteins: 13705 / 15499
Unmapped proteins: 1794


,GeneSym,ENSP
23,ZBTB44,ENSP00000404580
25,YPEL1,ENSP00000204279
28,WRB,ENSP00000327716
61,TTLL4,ENSP00000258398
63,TTC39B,ENSP00000347920


,GeneSym,ENSP,GeneID,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,P51523,P51523,P51523,UniProt_ID
0,ZNF627,ENSP00000354414,199692,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,Q7L945,Q7L945,Q7L945,Q7L945
1,ZNF544,ENSP00000269829,27300,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,Q6NX49,Q6NX49,Q6NX49,Q6NX49
2,ZNF540,ENSP00000324598,163255,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,Q8NDQ6,Q8NDQ6,Q8NDQ6,Q8NDQ6
3,ZNF512,ENSP00000347648,84450,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,Q96ME7,Q96ME7,Q96ME7,Q96ME7
4,ADAT1,ENSP00000310015,23536,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,Q9BUB4,Q9BUB4,Q9BUB4,Q9BUB4


Saved to TISSUES_with_UniProt.csv


In [19]:
conf_data

,GeneSym,ENSP,GeneID,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,P51523,P51523,P51523,UniProt_ID
0,ZNF627,ENSP00000354414,199692,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,Q7L945,Q7L945,Q7L945,Q7L945
1,ZNF544,ENSP00000269829,27300,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,Q6NX49,Q6NX49,Q6NX49,Q6NX49
2,ZNF540,ENSP00000324598,163255,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,Q8NDQ6,Q8NDQ6,Q8NDQ6,Q8NDQ6
3,ZNF512,ENSP00000347648,84450,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,Q96ME7,Q96ME7,Q96ME7,Q96ME7
4,ADAT1,ENSP00000310015,23536,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,Q9BUB4,Q9BUB4,Q9BUB4,Q9BUB4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15494,FGL2,ENSP00000248598,10875,0.0,0.0,0.0,0.0,0.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,Q14314
15495,CSTA,ENSP00000264474,1475,0.0,0.0,0.0,0.0,0.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,P01040
15496,S100A8,ENSP00000357721,6279,0.0,0.0,0.0,0.0,1.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,P05109
15497,HLA-DMB,ENSP00000378723,3109,0.0,0.0,0.0,0.0,0.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,P28068


UniProt_ID
Q9P1U0    1
O75312    1
Q86V21    1
Q5VST6    1
Q96I13    1
P51523    1
Q7L945    1
Q6NX49    1
Q8NDQ6    1
Q96ME7    1
Q9BUB4    1
O60304    1
Q9Y4E5    1
Q8TF68    1
Q9Y3M9    1
J3KSW0    1
Q9UDV6    1
P52747    1
P52739    1
Q14587    1
O75800    1
Q5VZL5    1
Q9H4I2    1
Q9HBF4    1
Q9NTW7    1
Name: count, dtype: int64

In [ ]:
conf_data_mapped = conf_data.dropna(subset=["UniProt_ID"]).reset_index(drop=True)
print(f"Retained: {len(conf_data_mapped)} rows")
conf_data_mapped.to_csv("TISSUES_with_UniProt_mapped.csv", index=False)
print("Saved to TISSUES_with_UniProt_mapped.csv")
conf_data_mapped
